# GAIL Procurement — Data Cleaning & Feature Engineering

Input: `po_master_anonymized.csv`, `cpbg_info_anonymized.csv`, `cpbg_intimation_diff_anonymized.csv`
(anonymized PO + CPBG data, FY 2023-24 to 2025-26)

This notebook:
1. Audits nulls and duplicate/redundant columns
2. Drops confirmed-redundant columns
3. Fixes data-quality issues found during EDA (SAP trailing-minus numbers, negative cycle times)
4. Engineers the features needed for every KPI, including three grounded in the
   GAIL C&P Procedure Manual (3rd Edition): CPBG value compliance (Clause 3.2.1A),
   CPBG duration compliance (Clause 3.2.1 Note ii), and intimation compliance
   (Clause 3.2.1 Note iii)
5. Saves cleaned, analysis-ready CSVs for the EDA notebook and Power BI


In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


## 1. Load raw data

In [ ]:
po = pd.read_csv("po_master_anonymized.csv")
cpbg = pd.read_csv("cpbg_info_anonymized.csv")
diff = pd.read_csv("cpbg_intimation_diff_anonymized.csv")

print("PO master:", po.shape)
print("CPBG info:", cpbg.shape)
print("CPBG intimation diff:", diff.shape)


## 2. Null audit

Flags any column with a meaningful null rate. Note: high nulls aren't always
"bad data" — e.g. most POs legitimately have no CPBG at all (orders under
₹5 lakhs are exempt per the C&P Procedure), so `BANK NAME.` / `BG NUMBER`
being ~83% null is expected sparsity, not an error.

In [ ]:
def null_audit(df, name):
    null_pct = (df.isnull().sum() / len(df) * 100).round(1)
    null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
    print(f"--- {name} ({df.shape[0]} rows) ---")
    for col, pct in null_pct.items():
        flag = "  <-- candidate drop (>80% null)" if pct > 80 else ""
        print(f"  {col:35s} {pct:5.1f}%{flag}")
    print()

null_audit(po, "PO_MASTER")
null_audit(cpbg, "CPBG_INFO")
null_audit(diff, "CPBG_DIFF")


## 3. Duplicate / redundant column check

Column-name similarity alone isn't proof of redundancy — checking whether
values in column A map 1:1 to column B is a much more reliable test.

In [ ]:
def check_1to1(df, a, b):
    sub = df[[a, b]].dropna()
    if len(sub) == 0:
        print(f"{a} vs {b}: no overlapping rows")
        return
    fwd = (sub.groupby(a)[b].nunique() == 1).mean() * 100
    bwd = (sub.groupby(b)[a].nunique() == 1).mean() * 100
    print(f"{a} -> {b}: {fwd:.1f}% consistent  |  {b} -> {a}: {bwd:.1f}% consistent")

for a, b in [("EIC", "EIC Name"), ("IND. DEPT.", "INDENTING DEPT."),
             ("Country Code", "Conutry"), ("PR Created by.", "PR created by Name."),
             ("Created by(CPF)", "Created by"), ("PLANT", "PLANT NAME")]:
    check_1to1(po, a, b)


**Result:** all pairs are confirmed duplicates (100% or near-100% both directions),
except `EIC`/`EIC Name` where the code is more reliable (100% code→name vs
only 37% name→code — the same name maps to multiple different codes, a real
data quality issue in the name field). Keep the code column in that case.

## 4. Drop confirmed-redundant / unusable columns

In [ ]:
po_drop_cols = [
    "SNO",                    # row index, not analytical
    "EIC Name",               # EIC (code) is the more reliable identifier
    "IND. DEPT.",             # 100% duplicate of INDENTING DEPT.
    "Conutry",                # 100% duplicate of Country Code (typo'd name)
    "PR created by Name.",    # ~100% duplicate of PR Created by.
    "Created by",             # 100% duplicate of Created by(CPF)
    "PLANT NAME",             # duplicate of PLANT
    "PO Ex Rate",             # Exchange Value(INR) already reflects conversion
]
po_clean = po.drop(columns=[c for c in po_drop_cols if c in po.columns])

print(f"TOTAL SCORE non-null: {po['TOTAL SCORE'].notna().sum()} / {len(po)}  "
      f"-- too sparse (98.5% null) to support a reliable KPI, keeping column but excluding from analysis")

cpbg_drop_cols = ["Assignment", "Transaction", "Entered On.1", "Dealing Officer"]
cpbg_clean = cpbg.drop(columns=[c for c in cpbg_drop_cols if c in cpbg.columns])

diff_drop_cols = ["BEYOND 30 DAYS"]  # 100% null and unreliable -- we derive our own flag
diff_clean = diff.drop(columns=[c for c in diff_drop_cols if c in diff.columns])

print("po_clean:", po_clean.shape, "| cpbg_clean:", cpbg_clean.shape, "| diff_clean:", diff_clean.shape)


## 5. Parse dates

In [ ]:
date_cols_po = ["PR Date", "PO DATE", "Schedule Delivery Da", "USER HOD Release DT", "C&P Release Date"]
for col in date_cols_po:
    po_clean[col] = pd.to_datetime(po_clean[col], errors="coerce")

date_cols_cpbg = ["PO Date", "Start of Term", "Term End", "FOI Date", "Intimation Date"]
for col in date_cols_cpbg:
    if col in cpbg_clean.columns:
        cpbg_clean[col] = pd.to_datetime(cpbg_clean[col], errors="coerce")


## 6. Clean SAP-style trailing-minus numbers

`D(LT-STD LT)` (lead time deviation) sometimes has values like `"29-"`
(SAP's way of writing -29) plus stray whitespace.

In [ ]:
def clean_trailing_minus(val):
    if pd.isna(val):
        return None
    s = str(val).strip()
    if s == "" or s.upper() in ("NIL", "NA", "N/A", "-"):
        return None
    if s.endswith("-"):
        s = "-" + s[:-1].strip()
    try:
        return float(s)
    except ValueError:
        return None

po_clean["lead_time_deviation_days"] = po_clean["D(LT-STD LT)"].apply(clean_trailing_minus)
po_clean["lead_time_deviation_days"].describe()


## 7. Feature engineering — lifecycle stages

Note: `USER HOD Release DT` and `C&P Release Date` were found to be
**identical in all 344 non-null rows** during EDA — a genuine data-capture
finding, not a bug. The funnel is simplified to 3 real stages instead of
showing a misleading "0-day" stage for HOD→C&P.

In [ ]:
po_clean["cycle_days_pr_to_po"] = (po_clean["PO DATE"] - po_clean["PR Date"]).dt.days
po_clean["approval_lag_days"] = (po_clean["PO DATE"] - po_clean["C&P Release Date"]).dt.days
po_clean["pr_to_approval_days"] = (po_clean["C&P Release Date"] - po_clean["PR Date"]).dt.days
po_clean["po_to_delivery_days"] = (po_clean["Schedule Delivery Da"] - po_clean["PO DATE"]).dt.days

# Flag logically-impossible values instead of silently averaging them in
po_clean["cycle_time_data_flag"] = np.where(po_clean["cycle_days_pr_to_po"] < 0, "NEGATIVE_CYCLE_CHECK", "OK")
po_clean["delivery_data_flag"] = np.where(po_clean["po_to_delivery_days"] < 0, "NEGATIVE_DELIVERY_CHECK", "OK")

print("Negative cycle time rows:", (po_clean["cycle_time_data_flag"] != "OK").sum())
print("Negative delivery rows:", (po_clean["delivery_data_flag"] != "OK").sum())


## 8. CPBG compliance features (grounded in the C&P Procedure)

- **Value compliance** (Clause 3.2.1A): CPBG must be 3% of order value for
  orders after 31-Mar-2023, 10% for orders before.
- **Duration compliance** (Clause 3.2.1 Note ii): guarantee term must cover
  the contract period + 90 days.
- Joins `cpbg_clean` to `po_clean` on `Purchasing Document` = `PO NO`
  (both anonymized consistently, so the join key still works).

In [ ]:
po_lookup = po_clean[["PO NO", "PO Basic Value (INR)", "Schedule Delivery Da", "BG VALUE"]].drop_duplicates("PO NO")
cpbg_clean = cpbg_clean.merge(po_lookup, left_on="Purchasing Document", right_on="PO NO", how="left")

print("Unmatched CPBG rows after join:", cpbg_clean["PO Basic Value (INR)"].isna().sum(), "/", len(cpbg_clean))

CUTOFF = pd.Timestamp("2023-04-01")
cpbg_clean["expected_cpbg_pct"] = np.where(cpbg_clean["PO Date"] < CUTOFF, 10.0, 3.0)
cpbg_clean["actual_cpbg_pct"] = cpbg_clean["Credit Amount"] / cpbg_clean["PO Basic Value (INR)"] * 100
cpbg_clean["cpbg_value_compliant"] = cpbg_clean["actual_cpbg_pct"] >= (cpbg_clean["expected_cpbg_pct"] - 0.5)

cpbg_clean["required_term_end"] = cpbg_clean["Schedule Delivery Da"] + pd.Timedelta(days=90)
cpbg_clean["cpbg_duration_compliant"] = cpbg_clean["Term End"] >= cpbg_clean["required_term_end"]

print("Value compliance rate:", round(cpbg_clean["cpbg_value_compliant"].mean()*100, 1), "%")
print("Duration compliance rate:", round(cpbg_clean["cpbg_duration_compliant"].mean()*100, 1), "%")


## 9. Intimation compliance (derived, not the unreliable manual flag)

The original `BEYOND 30 DAYS` column is 100% blank in the source data — we
derive the same 30-day rule (Clause 3.2.1 Note iii) directly from the numeric
`Difference` column instead.

In [ ]:
diff_clean["Difference_num"] = pd.to_numeric(diff_clean["Difference"], errors="coerce")
diff_clean["intimation_compliant"] = diff_clean["Difference_num"] <= 30

print("Intimation compliance rate:", round(diff_clean["intimation_compliant"].mean()*100, 1), "%")


## 10. Outlier scan (IQR method)

In [ ]:
def iqr_outliers(series, label):
    s = pd.to_numeric(series, errors="coerce").dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers = s[(s < lower) | (s > upper)]
    print(f"{label}: {len(outliers)}/{len(s)} outliers ({len(outliers)/len(s)*100:.1f}%), bounds=[{lower:.0f}, {upper:.0f}]")

iqr_outliers(po_clean["PO Basic Value (INR)"], "PO Basic Value (INR)")
iqr_outliers(cpbg_clean["Credit Amount"], "CPBG Credit Amount")
iqr_outliers(po_clean.loc[po_clean["cycle_time_data_flag"]=="OK", "cycle_days_pr_to_po"], "PR-to-PO cycle days (valid rows only)")


## 11. MSME category & FY-quarter derivation

In [ ]:
def msme_cat(v):
    if pd.isna(v):
        return "Others"
    v = str(v)
    if v.startswith("MICRO"):
        return "Micro"
    if v.startswith("SMALL"):
        return "Small"
    if v.startswith("MED"):
        return "Medium"
    return "Others"

po_clean["msme_category"] = po_clean["VENDOR TYPE"].apply(msme_cat)
po_clean["po_quarter"] = po_clean["PO DATE"].dt.to_period("Q").astype(str)

po_clean["msme_category"].value_counts()


## 12. Save cleaned, analysis-ready outputs

In [ ]:
po_clean.to_csv("po_master_cleaned.csv", index=False)
cpbg_clean.to_csv("cpbg_info_cleaned.csv", index=False)
diff_clean.to_csv("cpbg_intimation_diff_cleaned.csv", index=False)

print("Saved:")
print(f"  po_master_cleaned.csv:            {po_clean.shape}")
print(f"  cpbg_info_cleaned.csv:            {cpbg_clean.shape}")
print(f"  cpbg_intimation_diff_cleaned.csv: {diff_clean.shape}")


## Summary of cleaning decisions

| Decision | Reason |
|---|---|
| Dropped 8 columns from `po_master` | Confirmed 1:1 duplicates or non-analytical row IDs |
| Dropped 4 columns from `cpbg_info` | 100% null or redundant identifiers |
| Kept `TOTAL SCORE` but excluded from KPIs | 98.5% null — too sparse to trust |
| Kept `BANK NAME.`/`BG NUMBER` sparsity as-is | Expected — most POs are CPBG-exempt by policy |
| Flagged (not deleted) 6 negative-cycle-time rows | Logically impossible; needs a decision, not silent averaging |
| Simplified funnel to 3 stages | HOD Release and C&P Release dates are identical in 100% of rows |
| Cleaned SAP trailing-minus numbers | `"29-"` → `-29` |
| Derived 3 CPBG compliance flags from C&P Procedure clauses | Value (3%/10%), Duration (+90 days), Intimation (30 days) |

Next: open `02_eda_visualizations.ipynb` for the plotted analysis.